# ARS Session 1 — RFT on the Current Architecture

This notebook runs the D3 Session 1 sequence: generation validation, Gate 0, positive RFT rollout collection, Stage 1 RFT training, and paired pure-steering evaluation. The route is `[17, 20, 23, 26]`, the magnitude head is disabled (`m ≡ 1`), the Phi-2 backbone stays frozen, and training uses clean completion-token mean NLL. Stage 2 / redesigned SteerNet experiments are intentionally not part of this session.

Run cells from top to bottom on a Kaggle **Dual T4** accelerator. Choose smoke or production scale in the execution configuration cell. Evaluation uses greedy decoding and repetition penalty 1.0. Pipeline quality/runtime failures are recorded in the final status cell; dependent stages are skipped when prerequisites fail.

## 0. Configure Kaggle runtime and checkout

Set allocator and tokenizer environment variables before launching Python worker processes. The checkout is refreshed from the `development` branch so notebook commands match the committed code.

## Execution and scale configuration

Set `SMOKE_TEST=True` for the short end-to-end sanity path; leave it `False` for the full Session 1 run. Gate 0 and positive-rollout thresholds remain enforced as prerequisites: failures skip dependent work and are summarized at the end instead of raising out of the notebook.

In [ ]:
# ==========================================
# EXECUTION & SCALE CONFIGURATION
# ==========================================
import os
from datetime import datetime
SMOKE_TEST = False  # Set to True for a 5-min sanity run, False for production run
ATTENTION_IMPLEMENTATION = "sdpa"  # Eager T4 probe produced invalid sampled logits
os.environ["ARS_ATTENTION_IMPLEMENTATION"] = ATTENTION_IMPLEMENTATION
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
PHASE_STATUS = {
    "equivalence": "pending", "gate0": "pending",
    "collection": "pending", "training": "pending",
    "calibration": "pending", "benchmark": "pending",
}
GATE0_N_OVERRIDE = None  # Optional explicit Gate 0 size; set to 60 to run the full probe in smoke mode

if SMOKE_TEST:
    EQUIVALENCE_N = 10        # Two quick passes over a small probe
    GATE0_N = 10              # Small Gate 0 sample; threshold is diagnostic in smoke mode
    N_RFT_QUESTIONS = 40       # Quick sample across shards
    RFT_K = 2                  # Rollouts per question
    TRAIN_EPOCHS = 1           # 1 epoch for fast sanity
    CALIB_N = 16               # Small calibration sample
    BENCH_N_TEST = 30          # Small benchmark test subset
    BENCH_SHOTS = ["0"]        # Fast single condition
else:
    EQUIVALENCE_N = 100       # Full equivalence probe
    GATE0_N = 60               # Production Gate 0 sample
    N_RFT_QUESTIONS = 2000     # Full production sample
    RFT_K = 8                  # Full rollouts
    TRAIN_EPOCHS = 2           # Full training epochs
    CALIB_N = 128              # Standard calibration sample
    BENCH_N_TEST = 1319        # Full test set
    BENCH_SHOTS = ["0", "2"] # Full 0-shot and 2-shot

if GATE0_N_OVERRIDE is not None:
    if GATE0_N_OVERRIDE < 1:
        raise ValueError("GATE0_N_OVERRIDE must be positive")
    GATE0_N = int(GATE0_N_OVERRIDE)

print({
    "SMOKE_TEST": SMOKE_TEST,
    "EQUIVALENCE_N": EQUIVALENCE_N,
    "GATE0_N": GATE0_N,
    "GATE0_N_OVERRIDE": GATE0_N_OVERRIDE,
    "N_RFT_QUESTIONS": N_RFT_QUESTIONS,
    "RFT_K": RFT_K,
    "TRAIN_EPOCHS": TRAIN_EPOCHS,
    "CALIB_N": CALIB_N,
    "BENCH_N_TEST": BENCH_N_TEST,
    "BENCH_SHOTS": BENCH_SHOTS,
})

In [ ]:
import codecs
import json
import os
import selectors
import subprocess
import sys
import time
from pathlib import Path

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"

REPO_URL = "https://github.com/Abdullah182155/adaptive-residual-steering.git"
REPO_DIR = Path("/kaggle/working/adaptive-residual-steering")
BRANCH = "development"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print(f"Repository: {REPO_DIR} ({BRANCH})", flush=True)

def _emit_stream_line(state, line, *, force=False):
    """Print child output snapshots as normal notebook lines with a short throttle."""
    now = time.monotonic()
    if line and (force or now - state["last_update"] >= 1.0 or "100%" in line):
        print(f"{state['prefix']}{line}", flush=True)
        state["last_update"] = now


def _consume_stream_chunk(state, chunk):
    """Split child output on newline and tqdm carriage-return boundaries."""
    state["pending"] += state["decoder"].decode(chunk)
    while "\r" in state["pending"] or "\n" in state["pending"]:
        cr = state["pending"].find("\r") if "\r" in state["pending"] else len(state["pending"])
        lf = state["pending"].find("\n") if "\n" in state["pending"] else len(state["pending"])
        boundary = min(cr, lf)
        separator = state["pending"][boundary]
        line, state["pending"] = state["pending"][:boundary], state["pending"][boundary + 1:]
        _emit_stream_line(state, line, force=(separator == "\n" or "100%" in line))


def _finish_stream(state):
    state["pending"] += state["decoder"].decode(b"", final=True)
    _emit_stream_line(state, state["pending"], force=True)


def _start_live_process(command, env=None):
    return subprocess.Popen(
        command,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0,
    )


def run_live(command, *, env=None, prefix=""):
    """Run one child and render its tqdm refreshes as flushed notebook output."""
    run_parallel_live([(prefix, command, env)])


def run_parallel_live(jobs):
    """Run child commands concurrently and stream each labelled pipe in this thread."""
    selector = selectors.DefaultSelector()
    processes = []
    for prefix, command, env in jobs:
        process = _start_live_process(command, env)
        processes.append((prefix, command, process))
        state = {
            "prefix": prefix,
            "decoder": codecs.getincrementaldecoder("utf-8")(errors="replace"),
            "pending": "",
            "last_update": 0.0,
        }
        selector.register(process.stdout, selectors.EVENT_READ, state)

    while selector.get_map():
        for key, _ in selector.select(timeout=0.25):
            chunk = os.read(key.fileobj.fileno(), 4096)
            if chunk:
                _consume_stream_chunk(key.data, chunk)
            else:
                _finish_stream(key.data)
                selector.unregister(key.fileobj)
                key.fileobj.close()

    selector.close()
    failures = []
    for prefix, command, process in processes:
        return_code = process.wait()
        if return_code:
            failures.append(f"{prefix.strip()} exited with code {return_code}: {command}")
    if failures:
        raise RuntimeError("\n".join(failures))


## 1. Verify the Kaggle accelerator and repository tests

The two collection workers each get one T4. Their per-process CUDA device is remapped to `cuda:0` by `CUDA_VISIBLE_DEVICES`. Training/evaluation use one T4 and the configured micro-batch size of 2.

In [ ]:
gpu_query = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    check=True, capture_output=True, text=True,
)
gpu_rows = [line.strip() for line in gpu_query.stdout.splitlines() if line.strip()]
print("Visible GPUs:", gpu_rows)
if len(gpu_rows) < 2 or any("T4" not in row for row in gpu_rows[:2]):
    raise RuntimeError("Select Kaggle's Dual T4 accelerator before continuing")

subprocess.run([sys.executable, "-u", "-m", "pytest", "tests/", "-x", "-q"], check=True)

## 2. Assert the Stage 1 configuration

These checks make the session locks explicit before any model is loaded. `max_tok_len=768`, collection batch size 2, gradient accumulation 16, and no generation batching above 2 keep memory use bounded on T4.

In [ ]:
from src.configs.config import RSCConfig

cfg = RSCConfig()
assert cfg.fixed_layers == [17, 20, 23, 26]
assert cfg.steernet_use_mag_head is False  # m ≡ 1
assert cfg.use_gate is False
assert cfg.phase1_lr == 1.5e-4
assert cfg.objective == "nll"
assert cfg.max_tok_len == 768
assert cfg.batch_size == 2 and cfg.grad_accum == 16
assert cfg.generation_repetition_penalty == 1.0
cfg.validate()
print({
    "route": cfg.fixed_layers,
    "mag_head": cfg.steernet_use_mag_head,
    "loss": "positive completion token-mean NLL",
    "phase1_lr": cfg.phase1_lr,
    "max_tok_len": cfg.max_tok_len,
    "micro_batch": cfg.batch_size,
    "grad_accum": cfg.grad_accum,
    "use_gate": cfg.use_gate,
})

## 3. Step 1 — Batched vs. batch-1 generation equivalence

This creates `report/rft/generation_equivalence.json`. It compares B=2 with batch-1 as a diagnostic. Session 1 collection and benchmark both generate one question at a time, so this unused batched path does not block them. If you later enable batched evaluation, require this check to pass first.

In [ ]:
RFT_DIR = Path("report/rft") / RUN_ID
RFT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Starting equivalence probe: {EQUIVALENCE_N} questions", flush=True)
equivalence_command = [
    sys.executable, "-u", "scripts/run_rft_pipeline.py",
    "--stage", "equivalence",
    "--n_questions", str(EQUIVALENCE_N),
    "--output_dir", str(RFT_DIR),
]
try:
    run_live(equivalence_command, prefix="[equivalence] ")
    equivalence = json.loads((RFT_DIR / "generation_equivalence.json").read_text(encoding="utf-8"))
    PHASE_STATUS["equivalence"] = "passed" if equivalence["quality_gate_passed"] else "diagnostic_mismatch"
except (RuntimeError, OSError, ValueError, KeyError) as exc:
    PHASE_STATUS["equivalence"] = f"failed: {exc}"
print("Equivalence status:", PHASE_STATUS["equivalence"], flush=True)

## 4. Step 2 — Gate 0 sampling sanity probe

Gate 0 compares 8 sampled continuations at temperature 0.7 with paired greedy decoding on `GATE0_N` questions. The notebook records whether the 70% and paired-greedy criteria pass. If they fail, dependent RFT steps are marked skipped and the notebook continues to the final status cell.

In [ ]:
print(f"Starting Gate 0 probe on {GATE0_N} questions", flush=True)
gate0_command = [
    sys.executable, "-u", "scripts/run_rft_pipeline.py",
    "--stage", "gate0",
    "--n_questions", str(GATE0_N),
    "--k", "8",
    "--temperature", "0.7",
    "--output_dir", str(RFT_DIR),
]
gate0_command.append("--gate0_nonblocking")
try:
    run_live(gate0_command, prefix="[gate0] ")
    gate0 = json.loads((RFT_DIR / "gate0_stats.json").read_text(encoding="utf-8"))
    GATE0_OK = bool(gate0["quality_gate_passed"])
    PHASE_STATUS["gate0"] = "passed" if GATE0_OK else "quality_failed"
    print({key: gate0[key] for key in ("n", "pass_at_k_lenient", "greedy_lenient_accuracy", "pass_at_k_delta_vs_greedy_pp", "quality_gate_passed", "passed")})
except (RuntimeError, OSError, ValueError) as exc:
    GATE0_OK = False
    PHASE_STATUS["gate0"] = f"failed: {exc}"
    print("Gate 0 could not complete; dependent RFT stages will be skipped:", exc, flush=True)
if not GATE0_OK:
    print("Gate 0 did not meet its quality threshold; collection and training are skipped.", flush=True)

## 5. Step 3 — Collect positive RFT rollouts on both T4s

Each worker receives a disjoint shard of the same seeded configured-question sample and writes its own JSONL/stat files. Workers are launched as unbuffered processes with live output inherited by the Kaggle cell, so both per-shard progress bars remain visible. Each process sees one T4 and initializes CUDA after device selection. The notebook reports the combined verified-positive yield; production skips training below 15%, while smoke mode may train only when at least one verified positive exists.

In [ ]:
COLLECTION_OK = False
RFT_ROWS_READY = False
if not GATE0_OK:
    PHASE_STATUS["collection"] = "skipped_gate0_failed"
    print("Skipping rollout collection because Gate 0 did not pass.", flush=True)
else:
    collection_env = os.environ.copy()
    collection_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    collection_env["TOKENIZERS_PARALLELISM"] = "false"
    collection_env["PYTHONUNBUFFERED"] = "1"
    workers = []
    for shard, gpu in ((0, "0"), (1, "1")):
        env = collection_env.copy()
        env["CUDA_VISIBLE_DEVICES"] = gpu
        command = [
            sys.executable, "-u", "scripts/run_rft_pipeline.py",
            "--stage", "collect",
            "--n_questions", str(N_RFT_QUESTIONS),
            "--k", str(RFT_K),
            "--temperature", "0.7",
            "--n_shards", "2",
            "--shard", str(shard),
            "--output_dir", str(RFT_DIR),
            "--allow_low_yield",
        ]
        workers.append((f"[GPU{gpu}/shard{shard}] ", command, env))
    try:
        print("Starting both rollout shards; live progress follows:", flush=True)
        run_parallel_live(workers)
        shard_stats = [json.loads((RFT_DIR / f"shard{i}_stats.json").read_text(encoding="utf-8")) for i in (0, 1)]
        total_rollouts = 0
        positive_rollouts = 0
        for shard in (0, 1):
            with (RFT_DIR / f"shard{shard}.jsonl").open(encoding="utf-8") as stream:
                for line in stream:
                    record = json.loads(line)
                    total_rollouts += len(record["samples"])
                    positive_rollouts += sum(bool(sample.get("strict_ok")) for sample in record["samples"])
        positive_rate = positive_rollouts / max(total_rollouts, 1)
        RFT_ROWS_READY = positive_rollouts > 0 and (SMOKE_TEST or positive_rate >= 0.15)
        COLLECTION_OK = True
        PHASE_STATUS["collection"] = "ready" if RFT_ROWS_READY else "insufficient_positive_yield"
        print({"shard0": shard_stats[0], "shard1": shard_stats[1], "positive_rollouts": positive_rollouts, "positive_rate": positive_rate, "training_ready": RFT_ROWS_READY})
    except (RuntimeError, OSError, ValueError, KeyError) as exc:
        PHASE_STATUS["collection"] = f"failed: {exc}"
        print("Collection could not complete; training will be skipped:", exc, flush=True)

## 6. Step 4 — Train Stage 1 RFT

Training combines the two rollout shards and samples mixed 0/2/4-shot contexts. It uses the locked fixed route, identity magnitude, frozen Phi-2, and positive-completion token-mean NLL. The trainer uses micro-batch 2 with gradient accumulation 16, gradient checkpointing, and a maximum 768-token sequence length. The resulting adapter is `checkpoints/steernet.pt`.

In [ ]:
TRAIN_OK = False
if not RFT_ROWS_READY:
    PHASE_STATUS["training"] = "skipped_no_usable_positive_rollouts"
    print("Skipping RFT training: no positives or production yield is below 15%.", flush=True)
else:
    train_env = os.environ.copy()
    train_env["CUDA_VISIBLE_DEVICES"] = "0"
    train_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    train_env["TOKENIZERS_PARALLELISM"] = "false"
    print(f"Starting RFT training for {TRAIN_EPOCHS} epoch(s)", flush=True)
    try:
        run_live([
            sys.executable, "-u", "scripts/run_rft_pipeline.py",
            "--stage", "train",
            "--epochs", str(TRAIN_EPOCHS),
            "--checkpoint", "checkpoints/steernet.pt",
            "--output_dir", str(RFT_DIR),
        ], env=train_env, prefix="[train] ")
        TRAIN_OK = Path("checkpoints/steernet.pt").is_file()
        PHASE_STATUS["training"] = "passed" if TRAIN_OK else "failed_no_checkpoint"
    except (RuntimeError, OSError, ValueError) as exc:
        PHASE_STATUS["training"] = f"failed: {exc}"
        print("Training failed; calibration and steering benchmark will be skipped:", exc, flush=True)

## 7. Step 5 — Calibrate and evaluate pure steering vs. baseline

The deployment contract is calibrated from validation data using the trained checkpoint. The paired benchmark then evaluates baseline and the isolated `steer_only` arm on the configured GSM8K test subset and shot conditions. Evaluation is greedy, uses repetition penalty 1.0, one question per generation, and reloads the dedicated `steernet.pt` checkpoint. No gate/router arm is enabled.

In [ ]:
BENCH_RECORD = Path("report") / f"phase1_gsm8k_{BENCH_N_TEST}_{RUN_ID}.json"
CONTRACT_PATH = Path("report") / f"deployment_contract_{RUN_ID}.json"
BENCH_OK = False
CALIBRATION_OK = False
eval_env = os.environ.copy()
eval_env["CUDA_VISIBLE_DEVICES"] = "0"
eval_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
eval_env["TOKENIZERS_PARALLELISM"] = "false"
if not TRAIN_OK:
    PHASE_STATUS["calibration"] = "skipped_no_trained_checkpoint"
    PHASE_STATUS["benchmark"] = "skipped_no_trained_checkpoint"
    print("Skipping calibration and steering benchmark because training did not produce a checkpoint.", flush=True)
else:
    try:
        print(f"Starting S5 calibration on {CALIB_N} rows", flush=True)
        run_live([
            sys.executable, "-u", "scripts/run_ars_pipeline.py",
            "--stage", "S5",
            "--s_fixed", "17,20,23,26",
            "--calib_sel_n", str(CALIB_N),
            "--contract", str(CONTRACT_PATH),
        ], env=eval_env, prefix="[calibration] ")
        CALIBRATION_OK = CONTRACT_PATH.is_file()
        PHASE_STATUS["calibration"] = "passed" if CALIBRATION_OK else "failed_no_contract"
    except (RuntimeError, OSError, ValueError) as exc:
        PHASE_STATUS["calibration"] = f"failed: {exc}"
        print("Calibration failed; benchmark will be skipped:", exc, flush=True)
    if CALIBRATION_OK:
        try:
            print(f"Starting benchmark: n={BENCH_N_TEST}, shots={BENCH_SHOTS}", flush=True)
            run_live([
                sys.executable, "-u", "scripts/run_ars_pipeline.py",
                "--stage", "bench",
                "--s_fixed", "17,20,23,26",
                "--contract", str(CONTRACT_PATH),
                "--n_test", str(BENCH_N_TEST),
                "--shots", *BENCH_SHOTS,
                "--arms", "baseline,steer_only",
                "--record", str(BENCH_RECORD),
                "--report",
            ], env=eval_env, prefix="[benchmark] ")
            BENCH_OK = BENCH_RECORD.is_file()
            PHASE_STATUS["benchmark"] = "passed" if BENCH_OK else "failed_no_record"
        except (RuntimeError, OSError, ValueError) as exc:
            PHASE_STATUS["benchmark"] = f"failed: {exc}"
            print("Benchmark failed:", exc, flush=True)
    else:
        PHASE_STATUS["benchmark"] = "skipped_calibration_failed"
        print("Skipping benchmark because calibration did not produce a valid contract.", flush=True)

## 8. Review paired results and save artifacts

The JSON record contains per-question correctness and aggregate strict/lenient accuracy. This cell prints the configured-test-set lenient scores and the paired steer-only-minus-baseline delta for each shot condition.

In [ ]:
if BENCH_OK:
    record_path = BENCH_RECORD
    record = json.loads(record_path.read_text(encoding="utf-8"))
    summary = {}
    for shots, result in record["shots"].items():
        baseline = result["baseline"]
        steer_only = result["steer_only"]
        summary[shots] = {
            "n": baseline["n"],
            "baseline_lenient_accuracy": baseline["lenient_accuracy"],
            "steer_only_lenient_accuracy": steer_only["lenient_accuracy"],
            "paired_delta_pp": steer_only["lenient_accuracy"] - baseline["lenient_accuracy"],
            "expected_n": BENCH_N_TEST,
            "n_matches_config": baseline["n"] == steer_only["n"] == BENCH_N_TEST,
        }
    print(json.dumps(summary, indent=2))
    print("Artifacts:", record_path, Path("checkpoints/steernet.pt"), CONTRACT_PATH)
else:
    print("No benchmark record was produced; see phase statuses for the skip/failure reason.", flush=True)
print("Session 1 phase status:")
print(json.dumps(PHASE_STATUS, indent=2))